# **Libraries**


In [21]:
from langchain_neo4j import LLMGraphTransformer #  This class is used to extract nodes and relationships from text
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate # It can define system and user messages separately.
from langchain_community.document_loaders import PyPDFLoader # Used to load the PDF document to convert it later to document objects
from langchain_text_splitters import RecursiveCharacterTextSplitter # Used for splitting the documents into smaller chunks (Piece of text) for better processing and analysis

import ollama

from pyvis.network import Network
import os

import torch

# **LLMGraphTransformer Knowledge Graph**


In [22]:
doc_path = r"/home/trainee3/Desktop/GraphRAG/abel-and-his-great-adventure.pdf"

if doc_path: # Checking if the document path exist
    # Initializing the loader
    loader = PyPDFLoader(file_path=doc_path) # Loading the PDF document using PyPDFLoader and creating the document objects
    
    pages = loader.load() #  The pages is a List of document objects, where each document object represents a page in the PDF. 
    
    print(f"Loaded {len(pages)} document object from {doc_path}")
else:
    print("No document path provided.")

# Access content
print(len(pages))
#content = data[0].page_content # Reading the first page of the document 
#print(content[:]) # printing the content of the first page

Loaded 11 document object from /home/trainee3/Desktop/GraphRAG/abel-and-his-great-adventure.pdf
11


In [23]:
splitter = RecursiveCharacterTextSplitter( # Defining the splitter
    chunk_size=4000,      # characters per chunk — tune based on how slow/fast your model runs
    chunk_overlap=200     # each chunk shares its last 200 characters
)

chunks = splitter.split_documents(pages) # Creating chunks
"""
Takes your pages (the per-page Document list from PyPDFLoader) and runs the splitter on each one, producing a new, longer list of smaller Document chunks. 
Each output chunk still keeps its original metadata (like which page it came from), just with page_content broken into smaller pieces.
"""


print(f"Split into {len(chunks)} chunks")
print(chunks[1])

Split into 11 chunks
page_content='He was a tall man, somewhat ungainly of figure and homely of face. But his large, deep eyes of 
velvety nut-brown were very beautiful and marvellously bright and clear for a man of his age. 
He wore a little pointed, well-cared-for beard, innocent of gray; but his hair was grizzled, and 
altogether he had the appearance of a man who had passed through many sorrows which had 
marked his body as well as his soul. Looking at him, I doubted Mrs. Campbell’s conclusion that 
he had not “minded” giving up college. This man had given up much and felt it deeply; but he 
had outlived the pain and the blessing of sacrifice had come to him. His voice was very 
melodious and beautiful, and the brown hand he held out to me was peculiarly long and shapely 
and flexible.
We went out to the garden in the scented moist air of a maritime spring evening. Behind the 
garden was a cloudy pine wood; the house closed it in on the left, while in front and on the 
right a row 

In [24]:
model = ChatOllama(model="gpt-oss:20b") # ChatOllama connects LangChain to a model running locally through Ollama.
#graph_transformer = LLMGraphTransformer(llm=model) # This class uses a language model to convert unstructured text into a knowledge graph structure.

In [ ]:
#Another form of LLMGraphTransformer

graph_transformer = LLMGraphTransformer(
    llm=model,
    allowed_nodes = [
    "Person",
    "Animal",
    "Place",
    "Organization",
    "Object",
    "Plant",
    "Event",
    "Concept"
],
    allowed_relationships = [
    "MEMBER_OF",
    "LIVES_IN",
    "LOCATED_IN",
    "LOCATED_NEAR",
    "SIBLING_OF",
    "PARENT_OF",
    "FRIEND_OF",
    "CARES_FOR",
    "OWNS",
    "HAS_PET",
    "LIKES",
    "LOVES",
    "SAVES",
    "SENDS",
    "SELLS",
    "GIVES",
    "WORKS_AT",
    "ATTENDS",
    "LEAVES",
    "PARTICIPATES_IN",
    "DIES_AT",
    "BURIED_AT",
    "BELIEVES",
    "HAS_TRAIT"
],
#strict_mode=False  # keeps everything the model generates, constraints become suggestions only
)

"""The allowed_nodes and allowed_relationships parameters help prevent the model from inventing too many 
different graph labels. """


graph_documents = graph_transformer.convert_to_graph_documents(chunks)

In [ ]:
# documents = [Document(page_content=documents)] # LLMGraphTransformer expects a list of documents

#graph_documents = await graph_transformer.aconvert_to_graph_documents(documents) # GraphDocument is a class with three attributes: 1) Nodes: List[Node], 2) relationships: List[Relationship], 3) source: Document

In [26]:
all_nodes = []
all_relationships = []
for gd in graph_documents:
    all_nodes.extend(gd.nodes)
    all_relationships.extend(gd.relationships)

print(f"Total: {len(all_nodes)} nodes, {len(all_relationships)} relationships")

Total: 152 nodes, 19 relationships


In [27]:
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

Nodes:[Node(id='Abel Armstrong', type='Person', properties={}), Node(id='L. M. Montgomery', type='Person', properties={}), Node(id='Stillwater', type='Place', properties={}), Node(id='Board Of School Trustees', type='Organization', properties={}), Node(id='Harbour Shore', type='Place', properties={}), Node(id='Tamzine', type='Person', properties={}), Node(id='Mrs. Campbell', type='Person', properties={}), Node(id='Yaller Cat', type='Animal', properties={}), Node(id='Father', type='Person', properties={}), Node(id='Mother', type='Person', properties={}), Node(id='Garden', type='Place', properties={}), Node(id='Narrator', type='Person', properties={})]
Relationships:[Relationship(source=Node(id='Abel Armstrong', type='Person', properties={}), target=Node(id='Board Of School Trustees', type='Organization', properties={}), type='MEMBER_OF', properties={}), Relationship(source=Node(id='Abel Armstrong', type='Person', properties={}), target=Node(id='Tamzine', type='Person', properties={}), t

# **Knowledge Graph Visualization**


In [31]:
from pyvis.network import Network
import os

def visualize_graph(graph_documents):
    net = Network(height="1200px", width="100%", directed=True,
                  notebook=False, bgcolor="#222222", font_color="white")

    nodes = []
    relationships = []
    for gd in graph_documents:
        nodes.extend(gd.nodes)
        relationships.extend(gd.relationships)

    node_dict = {node.id: node for node in nodes}

    # --- Add ALL nodes, not just ones that appear in a relationship ---
    for node in nodes:
        try:
            net.add_node(node.id, label=node.id, title=node.type, group=node.type)
        except:
            continue

    # --- Add edges (unchanged logic — still only valid ones) ---
    for rel in relationships:
        if rel.source.id in node_dict and rel.target.id in node_dict:
            try:
                net.add_edge(rel.source.id, rel.target.id, label=rel.type.lower())
            except:
                continue

    net.set_options("""
    {
        "physics": {
            "forceAtlas2Based": {
                "gravitationalConstant": -100,
                "centralGravity": 0.01,
                "springLength": 200,
                "springConstant": 0.08
            },
            "minVelocity": 0.75,
            "solver": "forceAtlas2Based"
        }
    }
    """)

    output_file = "knowledge_graph.html"
    net.save_graph(output_file)
    print(f"Graph saved to {os.path.abspath(output_file)}")

    try:
        import webbrowser
        webbrowser.open(f"file://{os.path.abspath(output_file)}")
    except:
        print("Could not open browser automatically")

visualize_graph(graph_documents)

Graph saved to /home/trainee3/Desktop/GraphRAG/knowledge_graph.html


# **Extracting Json Files**


In [ ]:
"""import json

nodes = []
relationships = []

for gd in graph_documents:
    for node in gd.nodes:
        nodes.append({
            "id": node.id,
            "type": node.type,
            "properties": node.properties,
        })
    for rel in gd.relationships:
        relationships.append({
            "source": rel.source.id,
            "target": rel.target.id,
            "type": rel.type,
            "properties": rel.properties,
        })

with open("Storynodes.json", "w", encoding="utf-8") as f:
    json.dump(nodes, f, indent=2, ensure_ascii=False)

with open("Storyrelationships.json", "w", encoding="utf-8") as f:
    json.dump(relationships, f, indent=2, ensure_ascii=False)"""

In [32]:
import json

# --- Aggregate nodes and relationships across ALL chunks ---
all_nodes = []
all_relationships = []
for gd in graph_documents:
    all_nodes.extend(gd.nodes)
    all_relationships.extend(gd.relationships)

# --- Export nodes to JSON ---
nodes_data = [
    {
        "id": n.id,
        "type": n.type,
        "properties": n.properties or {}
    }
    for n in all_nodes
]

with open("Storynodes2.json", "w") as f:
    json.dump(nodes_data, f, indent=2)

# --- Export relationships to JSON ---
relationships_data = [
    {
        "source": r.source.id,   # flatten Node object down to just its id string
        "target": r.target.id,
        "type": r.type,
        "properties": r.properties or {}
    }
    for r in all_relationships
]

with open("Storyrelationships2.json", "w") as f:
    json.dump(relationships_data, f, indent=2)

print(f"Exported {len(nodes_data)} nodes to Storynodes2.json")
print(f"Exported {len(relationships_data)} relationships to Storyrelationships2.json")

Exported 152 nodes to Storynodes2.json
Exported 19 relationships to Storyrelationships2.json


In [23]:
import json
from collections import Counter

with open("Storynodes.json") as f:
    nodes = json.load(f)

pairs = [(n["id"], n["type"]) for n in nodes]
counts = Counter(pairs)
duplicates = {k: v for k, v in counts.items() if v > 1}

print(f"Total entries in JSON: {len(nodes)}")
print(f"Unique (id, type) pairs: {len(counts)}")
print(f"Number of duplicated pairs: {len(duplicates)}")
for k, v in list(duplicates.items())[:10]:
    print(k, "→ appears", v, "times")

Total entries in JSON: 377
Unique (id, type) pairs: 258
Number of duplicated pairs: 35
('Abel Armstrong', 'Person') → appears 2 times
('Stillwater', 'Place') → appears 4 times
('Narrator', 'Person') → appears 12 times
('Abel_Armstrong', 'Person') → appears 2 times
('Tamzine', 'Person') → appears 9 times
('Father', 'Person') → appears 4 times
('Family', 'Group') → appears 2 times
('Mrs. Campbell', 'Person') → appears 3 times
('Mother', 'Person') → appears 3 times
('Garden', 'Place') → appears 15 times


In [27]:
import json
from collections import defaultdict

with open("Storynodes.json") as f:
    nodes = json.load(f)

id_to_types = defaultdict(set)
for n in nodes:
    id_to_types[n["id"]].add(n["type"])

inconsistent = {id_: types for id_, types in id_to_types.items() if len(types) > 1}
print(f"Entity IDs with more than one type: {len(inconsistent)}")
for id_, types in list(inconsistent.items())[:10]:
    print(id_, "→", types)

Entity IDs with more than one type: 17
Garden → {'Thing', 'Place', 'Object', 'Location', 'Concept'}
Cat → {'Animal', 'Cat'}
Sky → {'Place', 'Concept'}
Lombardy Poplars → {'Object', 'Plant'}
Gate → {'Place', 'Object'}
White_Lilacs → {'Flower', 'Plant'}
Trellis → {'Object', 'Structure'}
Sea → {'Place', 'Concept'}
View → {'Object', 'Concept'}
Hop-Vine_Arbour → {'Place', 'Concept'}


# **KGGEN**


In [18]:
from kg_gen import KGGen

In [19]:
kg = KGGen(model="ollama/gpt-oss:20b")

In [20]:
from kg_gen import Graph

story_text = "\n\n".join(page.page_content for page in pages)  # KGGen expects a plain string, not Document objects

CHUNK_SIZE = 4000
CHUNK_OVERLAP = 200

# Steers extraction away from literal triple-mining of figurative prose (metaphors, imagery,
# abstract nouns like "soul" / "origin of evil") and toward the things we actually want in the
# graph: characters, places, animals, objects and the events/relations between them.
EXTRACTION_CONTEXT = (
    "This text is a work of literary fiction, not a factual or technical document. "
    "Extract only concrete entities: named people, animals, places, and physical objects, "
    "and the concrete events or relationships between them (e.g. who lives where, who owns "
    "what, who talks to whom, family relations, etc..). Resolve pronouns (he/she/I/you) to the "
    "character's name when it is clear from context. Do NOT extract abstract nouns, emotions, "
    "metaphors, or descriptive imagery (e.g. 'soul', 'origin of evil', 'warm colours') as entities."
)

# Chunking manually and looping sequentially, instead of kg.generate(chunk_size=...): that path
# fires every chunk concurrently via ThreadPoolExecutor, which just queues up against a single
# local Ollama GPU instance and times out. One request at a time matches how Ollama actually
# processes them, so nothing is wasted waiting.
chunk_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
text_chunks = chunk_splitter.split_text(story_text)
print(f"Split into {len(text_chunks)} chunks")

all_entities = set()
all_relations = set()

for i, chunk in enumerate(text_chunks):
    # gpt-oss:20b occasionally returns an empty response (reasoning-model quirk) and dspy raises
    # instead of returning an empty graph. Skip that one chunk rather than losing everything
    # accumulated so far in the loop.
    try:
        chunk_graph = kg.generate(input_data=chunk, context=EXTRACTION_CONTEXT, cluster=False)
    except Exception as e:
        print(f"Chunk {i + 1}/{len(text_chunks)} failed, skipping: {e}")
        continue

    all_entities.update(chunk_graph.entities)
    all_relations.update(chunk_graph.relations)
    print(f"Chunk {i + 1}/{len(text_chunks)} done | {len(all_entities)} entities so far")

graph = Graph(
    entities=all_entities,
    relations=all_relations,
    edges={r[1] for r in all_relations},
)
graph = kg.cluster(graph, context=EXTRACTION_CONTEXT)  # merges duplicate/coreferent entities

print(f"Entities: {len(graph.entities)}, Relations: {len(graph.relations)}")

Split into 11 chunks
Chunk 1/11 failed, skipping: LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: We need to extract key entities from the source text. Entities are subjects or objects. We need thorough list. Let's parse text. Entities include people, places, organizations, objects, etc. Let's list all names: Abel Armstrong, L. M. Montgomery (author), Stillwater (town), board of school trustees, schoolmaster, Tamzine (sister), Mrs. Campbell, the harbour shore, the garden, the cat, the mother’s people? Also ": "the new schoolmaster"  } 

Expected to find output fields in the LM response: [entities] 


Chunk 2/11 failed, skipping: LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: We need to extract key entities from the source text. Entities are subjects or objects. We need thorough list. Let's parse text. Entities include people, places, objects, 

KeyboardInterrupt: 

In [7]:
import json

nodes_data = [{"id": e, "type": "Entity", "properties": {}} for e in graph.entities]

with open("KGGEN2nodes.json", "w") as f:
    json.dump(nodes_data, f, indent=2)

# Relationships: (subject, predicate, object) tuples -> Cypher-friendly dicts
def to_rel_type(predicate):
    return predicate.upper().replace(" ", "_")

relationships_data = [
    {"source": s, "target": o, "type": to_rel_type(p), "properties": {}}
    for s, p, o in graph.relations
]

with open("KGGEN2relationships.json", "w") as f:
    json.dump(relationships_data, f, indent=2)

print(f"Exported {len(nodes_data)} nodes to KGGENnodes.json")
print(f"Exported {len(relationships_data)} relationships to KGGENrelationships.json")

if graph.entity_clusters:
    print("\nEntity clusters (canonical -> merged variants):")
    for canonical, variants in graph.entity_clusters.items():
        print(f"  {canonical}: {variants}")

Exported 68 nodes to KGGENnodes.json
Exported 170 relationships to KGGENrelationships.json
